In [ ]:
import ibis
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from pathlib import Path

table_in_name = "working_fixed"
table_out_name = "spatial_distance_matrix"

# ==========================================
# 1. SETUP & EXTENSION LOADING
# ==========================================
dirs = get_data_dirs()
print("⏳ Initializing DuckDB connection and loading spatial extension...")
start_time = pd.Timestamp.now()

# Assuming dirs.db_path is defined
con = ibis.duckdb.connect(dirs.db_path)
con.raw_sql("INSTALL spatial; LOAD spatial;")

print(f"✅ Database connected in {pd.Timestamp.now() - start_time:.1f} seconds.\n")

# ==========================================
# 2. THE IBIS ORM PIPELINE (AST Building)
# ==========================================
print("⏳ Building Ibis relational algebra (Lazy Evaluation)...")
ast_start = pd.Timestamp.now()

t = con.table(table_in_name)

# Filter non-nulls 
t_clean = t.filter([
    t.lon_dec.notnull(), 
    t.lat_dec.notnull(), 
    t.pc4.notnull()
])

# Aliases for the Self-Join
left = t_clean.alias("a")
right = t_clean.alias("b")

# Block-Diagonal Join & Diagonal Filter 
joined = left.join(
    right,
    [
        left.pc4 == right.pc4,                             # Block-Diagonal
        left.registered_number != right.registered_number  # Exclude self-pairs
    ]
)

# Project the required columns
pairs = joined.select(
    firm_i = left.registered_number,
    firm_j = right.registered_number,
    pc4 = left.pc4,
    lon_i = left.lon_dec,
    lat_i = left.lat_dec,
    lon_j = right.lon_dec,
    lat_j = right.lat_dec
)

# Register the Ibis logic as a virtual table (No data processed yet)
con.create_view("temp_spatial_pairs", pairs, overwrite=True)
print(f"✅ Ibis AST built and view registered in {pd.Timestamp.now() - ast_start:.1f} seconds.\n")

# ==========================================
# 3. RAW SQL SPATIAL WRAPPER & MATERIALIZATION
# ==========================================
print("⏳ Starting out-of-core physical materialization to disk. This may take a minute...")
exec_start = pd.Timestamp.now()

# Apply the ST_Point and ST_Distance_Sphere logic to the Ibis view.
spatial_matrix_expr = con.sql("""
    SELECT 
        firm_i,
        firm_j,
        pc4,
        ST_Distance_Sphere(
            ST_Point(lon_i, lat_i), 
            ST_Point(lon_j, lat_j)
        ) AS distance_meters
    FROM temp_spatial_pairs
""")

# Execute the entire unified plan out-of-core
con.create_table(table_out_name, spatial_matrix_expr, overwrite=True)

exec_duration = pd.Timestamp.now() - exec_start
print(f"✅ Materialization complete in {exec_duration:.1f} seconds.\n")

# ==========================================
# 4. VERIFICATION
# ==========================================
print("⏳ Verifying output...")
row_count = con.table(table_out_name).count().execute()

print(f"🎉 SUCCESS! Total edges calculated: {row_count:,}")
print(f"⏱️ Total script execution time: {pd.Timestamp.now() - start_time:.1f} seconds.\n")

# Optional: Peek at the data
display(con.table(table_out_name).head(5).execute())